In [391]:
import os
import subprocess
import json

# Optional: if you want to force a specific GPU manually, set this to an integer like 1
FORCE_GPU_ID = None

# Maximum utilization (%) for a GPU to be considered idle
MAX_UTIL_PERCENT = 10

# Minimum free memory (MiB) required to consider a GPU usable
MIN_FREE_MEM_MIB = 10000

def query_gpus():
    cmd = [
        "nvidia-smi",
        "--query-gpu=index,memory.total,memory.used,memory.free,utilization.gpu",
        "--format=csv,noheader,nounits"
    ]
    out = subprocess.check_output(cmd, text=True).strip().splitlines()

    gpus = []
    for line in out:
        idx, total, used, free, util = [x.strip() for x in line.split(",")]
        gpus.append({
            "index": int(idx),
            "total_mib": int(total),
            "used_mib": int(used),
            "free_mib": int(free),
            "util_percent": int(util),
        })
    return gpus

def pick_best_gpu(gpus, max_util_percent=10, min_free_mem_mib=20000):
    # Step 1: filter GPUs with utilization < max_util_percent
    idle = [g for g in gpus if g["util_percent"] < max_util_percent]

    if not idle:
        raise RuntimeError(
            f"No GPU has utilization below {max_util_percent}%. "
            f"Available: {json.dumps(gpus, indent=2)}"
        )

    # Step 2: among idle GPUs, filter those with enough free memory
    usable = [g for g in idle if g["free_mib"] >= min_free_mem_mib]

    if not usable:
        raise RuntimeError(
            f"No idle GPU (<{max_util_percent}% util) has at least {min_free_mem_mib} MiB free. "
            f"Idle GPUs: {json.dumps(idle, indent=2)}"
        )

    # Step 3: pick the one with the most free memory
    return max(usable, key=lambda x: x["free_mib"])

gpus = query_gpus()

print("Visible GPU status:")
for g in gpus:
    print(
        f"GPU {g['index']}: free={g['free_mib']} MiB | "
        f"used={g['used_mib']} MiB / {g['total_mib']} MiB | "
        f"util={g['util_percent']}%"
    )

if FORCE_GPU_ID is not None:
    chosen_gpu = int(FORCE_GPU_ID)
    print(f"\nFORCE_GPU_ID is set. Using GPU {chosen_gpu}.")
else:
    best = pick_best_gpu(gpus, max_util_percent=MAX_UTIL_PERCENT, min_free_mem_mib=MIN_FREE_MEM_MIB)
    chosen_gpu = best["index"]
    print(f"\nAuto-selected GPU {chosen_gpu} with {best['free_mib']} MiB free and {best['util_percent']}% utilization.")

# Set BEFORE torch import
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = str(chosen_gpu)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("\nEnvironment set:")
print("CUDA_VISIBLE_DEVICES =", os.environ["CUDA_VISIBLE_DEVICES"])
print("PYTORCH_CUDA_ALLOC_CONF =", os.environ["PYTORCH_CUDA_ALLOC_CONF"])
print("\nIMPORTANT: If torch was already imported earlier in this kernel, restart the kernel and run from Cell 1 again.")
print("Project by Harissh Krishna L\nSabareesh S and\nSharavan Kumar N")


Visible GPU status:
GPU 0: free=6343 MiB | used=136824 MiB / 143771 MiB | util=2%
GPU 1: free=1910 MiB | used=141258 MiB / 143771 MiB | util=23%
GPU 2: free=80786 MiB | used=62382 MiB / 143771 MiB | util=24%
GPU 3: free=113 MiB | used=143055 MiB / 143771 MiB | util=0%
GPU 4: free=82065 MiB | used=61103 MiB / 143771 MiB | util=22%
GPU 5: free=18970 MiB | used=124197 MiB / 143771 MiB | util=0%
GPU 6: free=3253 MiB | used=139914 MiB / 143771 MiB | util=0%
GPU 7: free=11999 MiB | used=131169 MiB / 143771 MiB | util=4%

Auto-selected GPU 5 with 18970 MiB free and 0% utilization.

Environment set:
CUDA_VISIBLE_DEVICES = 5
PYTORCH_CUDA_ALLOC_CONF = expandable_segments:True

IMPORTANT: If torch was already imported earlier in this kernel, restart the kernel and run from Cell 1 again.
Project by Harissh Krishna L
Sabareesh S and
Sharavan Kumar N


In [392]:
import os
import gc
import json
import math
import copy
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForMaskedLM
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

try:
    from IPython.display import display
except Exception:
    display = print

In [393]:
def clear_gpu_memory():
    """
    Clears the GPU cache and triggers Python garbage collection
    to free up memory for the next seed/task.
    """
    # 1. Clear out any unreferenced objects in Python
    gc.collect()
    
    # 2. Clear the PyTorch cache across all available GPUs
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        # If you want to be extra thorough on a multi-GPU DGX:
        torch.cuda.ipc_collect() 
        
    print(f"GPU Cache Cleared. Current Memory Allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

In [394]:
clear_gpu_memory()

GPU Cache Cleared. Current Memory Allocated: 1235.77 MB


In [395]:
import os
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))


CUDA_VISIBLE_DEVICES = 5


In [396]:
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

if torch.cuda.is_available():
    print("Visible CUDA device count:", torch.cuda.device_count())
    print("Current visible device index:", torch.cuda.current_device())
    print("Visible device 0 name:", torch.cuda.get_device_name(0))


DEVICE: cuda:0
Visible CUDA device count: 1
Current visible device index: 0
Visible device 0 name: NVIDIA H200


In [397]:
GLOBAL_CONFIG = {
    # -------- runtime --------
    "seed": 13,
    "shots_per_class": 25,
    "n_subsets": 8,

    # -------- model --------
    "english_model_name": "bert-base-uncased",
    "chinese_model_name": "bert-base-chinese",
    "max_length": 256,
    "num_soft_tokens": 15,
    "dropout": 0.45,
    "freeze_plm": False,

    # -------- optimization --------
    "lr": 3e-5,
    "refinement_lr": 3e-5,
    "weight_decay": 1e-4,
    "l2_alpha": 1e-3,
    "batch_size": 32,
    "accumulation_steps": 1,
    "source_epochs": 20,
    "refinement_epochs": 8,
    "final_epochs": 20,
    "grad_clip": 1.0,
    "amp": True,

    # -------- dataloader --------
    "num_workers": 0,
    "pin_memory": False,

    # -------- error propagation safeguards --------
    "ema_decay": 0.975,
    "anchor_alpha": 2e-3,

    # -------- pseudo-label stabilization --------
    "prediction_temperature": 0.7,
    "train_refine_conf_threshold": 0.9,
    "final_invariant_conf_threshold": 0.95,
    "use_source_replay_in_refinement": True,
    "source_replay_factor_refine": 6,
    "source_replay_factor_final": 5,
    "required_vote_ratio":1,

    # -------- CNS voting & adaptive EMA --------
    "cns_beta": 0.6,
    "ema_decay_min": 0.95,
    "ema_decay_max": 0.9999,
    "ema_adapt_gamma": 1.5,
    "ema_tau_up": 0.015,
    "ema_tau_down": 0.01,

    # -------- early stopping --------
    "early_stop_source_threshold": 1.0,
    "early_stop_refine_threshold": 0.95,
    "early_stop_final_threshold": 0.97,
    "early_stop_patience": 2,

    # -------- prompt / verbalizer --------
    "label_map": {0: "Human", 1: "AI"},
    "prompt_text": {
        "english": "This text was written by",
        "chinese": "这段文本由",
    },
    "verbalizer": {
        "english": {
            0: ["human", "person", "people"],
            1: ["ai", "llm", "large language model"],
        },
        "chinese": {
            0: ["医生", "大夫", "专家"],
            1: ["机器", "智能", "生成"],
        },
    },

    # -------- save dirs --------
    "save_root": "/nfsshare/users/P127003093/Mini Project/output",
    "save_reports_dir":     "/nfsshare/users/P127003093/Mini Project/output/reports",
    "save_histories_dir":   "/nfsshare/users/P127003093/Mini Project/output/histories",
    "save_checkpoints_dir": "/nfsshare/users/P127003093/Mini Project/output/checkpoints",
    "save_votes_dir":       "/nfsshare/users/P127003093/Mini Project/output/votes",
    "save_configs_dir":     "/nfsshare/users/P127003093/Mini Project/output/configs",
}

for k in [
    "save_root",
    "save_reports_dir",
    "save_votes_dir",
    "save_histories_dir",
    "save_checkpoints_dir",
    "save_configs_dir",
]:
    os.makedirs(GLOBAL_CONFIG[k], exist_ok=True)


In [398]:
CHEAT_ROOT = "/nfsshare/users/P127003093/Mini Project/datasets/CHEAT"
MEDQA_ROOT = "/nfsshare/users/P127003093/Mini Project/datasets/Chinese Medical QnA"

In [399]:
TASK_REGISTRY = {
    # -------- CHEAT --------
    "A->M": {
        "source_file": os.path.join(CHEAT_ROOT, "Algorithm.csv"),
        "target_file": os.path.join(CHEAT_ROOT, "Medicine.csv"),
        "dataset_type": "english",
    },
    "A->S": {
        "source_file": os.path.join(CHEAT_ROOT, "Algorithm.csv"),
        "target_file": os.path.join(CHEAT_ROOT, "Subject.csv"),
        "dataset_type": "english",
    },
    "M->A": {
        "source_file": os.path.join(CHEAT_ROOT, "Medicine.csv"),
        "target_file": os.path.join(CHEAT_ROOT, "Algorithm.csv"),
        "dataset_type": "english",
    },
    "M->S": {
        "source_file": os.path.join(CHEAT_ROOT, "Medicine.csv"),
        "target_file": os.path.join(CHEAT_ROOT, "Subject.csv"),
        "dataset_type": "english",
    },
    "S->A": {
        "source_file": os.path.join(CHEAT_ROOT, "Subject.csv"),
        "target_file": os.path.join(CHEAT_ROOT, "Algorithm.csv"),
        "dataset_type": "english",
    },
    "S->M": {
        "source_file": os.path.join(CHEAT_ROOT, "Subject.csv"),
        "target_file": os.path.join(CHEAT_ROOT, "Medicine.csv"),
        "dataset_type": "english",
    },

    # -------- Chinese Medical QA --------
    "I->O": {
        "source_file": os.path.join(MEDQA_ROOT, "IM_2000_AI_HUMAN.csv"),
        "target_file": os.path.join(MEDQA_ROOT, "Oncology_2000_AI_HUMAN.csv"),
        "dataset_type": "chinese",
    },
    "I->P": {
        "source_file": os.path.join(MEDQA_ROOT, "IM_2000_AI_HUMAN.csv"),
        "target_file": os.path.join(MEDQA_ROOT, "Pediatrics_2000_AI_HUMAN.csv"),
        "dataset_type": "chinese",
    },
    "O->I": {
        "source_file": os.path.join(MEDQA_ROOT, "Oncology_2000_AI_HUMAN.csv"),
        "target_file": os.path.join(MEDQA_ROOT, "IM_2000_AI_HUMAN.csv"),
        "dataset_type": "chinese",
    },
    "O->P": {
        "source_file": os.path.join(MEDQA_ROOT, "Oncology_2000_AI_HUMAN.csv"),
        "target_file": os.path.join(MEDQA_ROOT, "Pediatrics_2000_AI_HUMAN.csv"),
        "dataset_type": "chinese",
    },
    "P->I": {
        "source_file": os.path.join(MEDQA_ROOT, "Pediatrics_2000_AI_HUMAN.csv"),
        "target_file": os.path.join(MEDQA_ROOT, "IM_2000_AI_HUMAN.csv"),
        "dataset_type": "chinese",
    },
    "P->O": {
        "source_file": os.path.join(MEDQA_ROOT, "Pediatrics_2000_AI_HUMAN.csv"),
        "target_file": os.path.join(MEDQA_ROOT, "Oncology_2000_AI_HUMAN.csv"),
        "dataset_type": "chinese",
    },
}

In [400]:
print("Registered tasks:")
for k, v in TASK_REGISTRY.items():
    print(k, "->", v["dataset_type"], "|", v["source_file"], "=>", v["target_file"])

Registered tasks:
A->M -> english | /nfsshare/users/P127003093/Mini Project/datasets/CHEAT/Algorithm.csv => /nfsshare/users/P127003093/Mini Project/datasets/CHEAT/Medicine.csv
A->S -> english | /nfsshare/users/P127003093/Mini Project/datasets/CHEAT/Algorithm.csv => /nfsshare/users/P127003093/Mini Project/datasets/CHEAT/Subject.csv
M->A -> english | /nfsshare/users/P127003093/Mini Project/datasets/CHEAT/Medicine.csv => /nfsshare/users/P127003093/Mini Project/datasets/CHEAT/Algorithm.csv
M->S -> english | /nfsshare/users/P127003093/Mini Project/datasets/CHEAT/Medicine.csv => /nfsshare/users/P127003093/Mini Project/datasets/CHEAT/Subject.csv
S->A -> english | /nfsshare/users/P127003093/Mini Project/datasets/CHEAT/Subject.csv => /nfsshare/users/P127003093/Mini Project/datasets/CHEAT/Algorithm.csv
S->M -> english | /nfsshare/users/P127003093/Mini Project/datasets/CHEAT/Subject.csv => /nfsshare/users/P127003093/Mini Project/datasets/CHEAT/Medicine.csv
I->O -> chinese | /nfsshare/users/P12700

In [401]:
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [402]:
def gpu_mem_text():
    if not torch.cuda.is_available():
        return "cpu"
    allocated = torch.cuda.memory_allocated() / (1024 ** 3)
    reserved = torch.cuda.memory_reserved() / (1024 ** 3)
    return f"{allocated:.2f}G/{reserved:.2f}G"

In [403]:
def stage_banner(task_name: str, stage_name: str, extra: str = ""):
    print("\n" + "=" * 120)
    print(f"[TASK={task_name}] [{stage_name}] DEVICE={DEVICE} GPU_MEM={gpu_mem_text()} {extra}")
    print("=" * 120)

In [404]:
def task_prefix(task_name: str, seed: int) -> str:
    return f"{task_name.replace('->', '_')}_{seed}"

In [405]:

def save_csv(df: pd.DataFrame, path: str):
    df.to_csv(path, index=False)
    return path

In [406]:
def save_json(obj: dict, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    return path


In [407]:
def clean_text(x: str) -> str:
    return " ".join(str(x).split()).strip()


In [408]:
def load_domain_csv(filename: str) -> pd.DataFrame:
    df = pd.read_csv(filename).copy()
    return df.reset_index(drop=True)


In [409]:
def infer_model_name(dataset_type: str) -> str:
    if dataset_type == "english":
        return GLOBAL_CONFIG["english_model_name"]
    elif dataset_type == "chinese":
        return GLOBAL_CONFIG["chinese_model_name"]
    raise ValueError(f"Unsupported dataset_type={dataset_type}")

In [410]:
def build_model_text(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    cols = set(df.columns)

    if {"title", "keyword", "abstract"}.issubset(cols):
        # CHEAT dataset: only the abstract is AI/human generated.
        # Title and keyword are always human-written and would confuse the detector,
        # so we feed only the abstract as model_text.
        df["model_text"] = df["abstract"].fillna("").astype(str).map(clean_text)
    elif {"dept", "title", "question", "answer"}.issubset(cols):
        df["model_text"] = (
            "Department: " + df["dept"].fillna("").astype(str).map(clean_text) + " "
            "Title: " + df["title"].fillna("").astype(str).map(clean_text) + " "
            "Question: " + df["question"].fillna("").astype(str).map(clean_text) + " "
            "Answer: " + df["answer"].fillna("").astype(str).map(clean_text)
        )
    elif "abstract" in cols:
        df["model_text"] = df["abstract"].fillna("").astype(str).map(clean_text)
    elif "answer" in cols:
        df["model_text"] = df["answer"].fillna("").astype(str).map(clean_text)
    else:
        raise ValueError(f"Unsupported columns: {df.columns.tolist()}")

    if "label" not in df.columns:
        raise ValueError("CSV must contain a 'label' column.")

    df["label"] = df["label"].astype(int)
    label_values = sorted(df["label"].dropna().unique().tolist())
    if not set(label_values).issubset({0, 1}):
        raise ValueError(f"Labels must be subset of {{0,1}}. Found: {label_values}")

    return df.reset_index(drop=True)

In [411]:
def sample_few_shot_per_class(df: pd.DataFrame, shots_per_class: int, seed: int) -> pd.DataFrame:
    pieces = []
    for label in sorted(df["label"].unique()):
        part = df[df["label"] == label].sample(n=shots_per_class, random_state=seed)
        pieces.append(part)
    out = pd.concat(pieces, axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out

In [412]:
def inspect_task_data(task_name: str, seed: int):
    shots_per_class = GLOBAL_CONFIG["shots_per_class"]
    spec = TASK_REGISTRY[task_name]
    source_df = build_model_text(load_domain_csv(spec["source_file"]))
    target_df = build_model_text(load_domain_csv(spec["target_file"]))

    print(f"\nTask: {task_name}")
    print("Dataset type:", spec["dataset_type"])
    print("Source shape:", source_df.shape)
    print("Target shape:", target_df.shape)
    print("Source labels:\n", source_df["label"].value_counts().sort_index())
    print("Target labels:\n", target_df["label"].value_counts().sort_index())

    few_df = sample_few_shot_per_class(source_df, shots_per_class, seed)
    print("Few-shot source shape:", few_df.shape)
    print("Few-shot label counts:\n", few_df["label"].value_counts().sort_index())

    display(source_df.head(2))
    display(target_df.head(2))

In [413]:
def load_task_data(task_name: str, seed: int):
    shots_per_class = GLOBAL_CONFIG["shots_per_class"]
    spec = TASK_REGISTRY[task_name]
    source_df = build_model_text(load_domain_csv(spec["source_file"]))
    target_df = build_model_text(load_domain_csv(spec["target_file"]))
    source_few_df = sample_few_shot_per_class(source_df, shots_per_class, seed)
    target_df = target_df.reset_index(drop=True).copy()
    target_df["gold_label"] = target_df["label"].astype(int)
    return source_few_df, target_df, spec["dataset_type"]


In [414]:
def resolve_phrase_verbalizer(tokenizer, verbalizer: Dict[int, List[str]]) -> Dict[int, List[List[int]]]:
    resolved = {}
    for label, phrases in verbalizer.items():
        ids_list = []
        for phrase in phrases:
            ids = tokenizer.encode(phrase, add_special_tokens=False)
            if len(ids) == 0:
                raise ValueError(f"Phrase tokenized to empty sequence: {phrase}")
            ids_list.append(ids)
        resolved[label] = ids_list
    return resolved

In [415]:
@dataclass
class MultiMaskSoftPromptCollator:
    tokenizer: AutoTokenizer
    max_length: int
    num_label_masks: int
    prompt_text: str

    def __call__(self, batch):
        input_ids_list = []
        attention_mask_list = []
        mask_positions_list = []
        labels = []
        idxs = []

        prompt_ids = self.tokenizer.encode(self.prompt_text, add_special_tokens=False)
        period_ids = self.tokenizer.encode(".", add_special_tokens=False)
        cls_id = self.tokenizer.cls_token_id
        sep_id = self.tokenizer.sep_token_id
        pad_id = self.tokenizer.pad_token_id
        mask_id = self.tokenizer.mask_token_id

        reserved = 1 + len(prompt_ids) + self.num_label_masks + len(period_ids) + 1
        text_budget = self.max_length - reserved
        if text_budget <= 0:
            raise ValueError("max_length is too small for prompt construction.")

        for item in batch:
            text = clean_text(item["text"])
            text_ids = self.tokenizer.encode(
                text,
                add_special_tokens=False,
                truncation=True,
                max_length=text_budget,
            )

            input_ids = [cls_id] + text_ids + prompt_ids + ([mask_id] * self.num_label_masks) + period_ids + [sep_id]
            mask_positions = [i for i, x in enumerate(input_ids) if x == mask_id]
            if len(mask_positions) != self.num_label_masks:
                raise ValueError("Mask positions could not be constructed correctly.")

            attention_mask = [1] * len(input_ids)
            pad_len = self.max_length - len(input_ids)
            input_ids += [pad_id] * pad_len
            attention_mask += [0] * pad_len

            input_ids_list.append(input_ids)
            attention_mask_list.append(attention_mask)
            mask_positions_list.append(mask_positions)
            idxs.append(item["idx"])
            if "label" in item:
                labels.append(int(item["label"]))

        out = {
            "input_ids": torch.tensor(input_ids_list, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask_list, dtype=torch.long),
            "mask_positions": torch.tensor(mask_positions_list, dtype=torch.long),
            "idxs": torch.tensor(idxs, dtype=torch.long),
        }
        if len(labels) > 0:
            out["labels"] = torch.tensor(labels, dtype=torch.long)
        return out

In [416]:
def build_tokenizer_and_collator(dataset_type: str):
    model_name = infer_model_name(dataset_type)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    verbalizer = GLOBAL_CONFIG["verbalizer"][dataset_type]
    resolved_verbalizer = resolve_phrase_verbalizer(tokenizer, verbalizer)
    num_label_masks = max(len(ids) for phrases in resolved_verbalizer.values() for ids in phrases)
    collator = MultiMaskSoftPromptCollator(
        tokenizer=tokenizer,
        max_length=GLOBAL_CONFIG["max_length"],
        num_label_masks=num_label_masks,
        prompt_text=GLOBAL_CONFIG["prompt_text"][dataset_type],
    )
    return tokenizer, resolved_verbalizer, num_label_masks, collator


In [417]:
class TextFrameDataset(Dataset):
    def __init__(self, df: pd.DataFrame, text_col: str = "model_text", label_col: str = "label", labeled: bool = True):
        self.df = df.reset_index(drop=True).copy()
        self.texts = self.df[text_col].fillna("").astype(str).tolist()
        self.labeled = labeled
        self.labels = self.df[label_col].astype(int).tolist() if labeled else None

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        item = {"text": self.texts[idx], "idx": idx}
        if self.labeled:
            item["label"] = self.labels[idx]
        return item

In [418]:
class PaperSoftPromptDetector(nn.Module):
    def __init__(
        self,
        model_name: str,
        tokenizer,
        resolved_verbalizer: Dict[int, List[List[int]]],
        num_soft_tokens: int = 10,
        dropout: float = 0.5,
        freeze_plm: bool = False,
    ):
        super().__init__()
        self.tokenizer = tokenizer
        self.resolved_verbalizer = resolved_verbalizer
        self.num_soft_tokens = num_soft_tokens

        self.mlm = AutoModelForMaskedLM.from_pretrained(model_name)
        self.hidden_size = self.mlm.config.hidden_size
        self.soft_prompt = nn.Parameter(torch.randn(num_soft_tokens, self.hidden_size) * 0.02)
        self.dropout = nn.Dropout(dropout)

        self.register_buffer("soft_prompt_attention", torch.ones(num_soft_tokens, dtype=torch.long))

        if freeze_plm:
            for p in self.mlm.parameters():
                p.requires_grad = False

    def _get_backbone(self):
        if hasattr(self.mlm, "bert"):
            return self.mlm.bert
        if hasattr(self.mlm, "roberta"):
            return self.mlm.roberta
        raise ValueError("This implementation expects a BERT or RoBERTa masked LM backbone.")

    def _get_mask_logits(self, vocab_logits, mask_positions):
        batch_size, num_masks = mask_positions.shape
        batch_idx = torch.arange(batch_size, device=vocab_logits.device).unsqueeze(1).expand(batch_size, num_masks)
        return vocab_logits[batch_idx, mask_positions, :]

    def _phrase_probability(self, mask_probs, token_ids: List[int]):
        phrase_len = len(token_ids)
        token_ids = torch.tensor(token_ids, device=mask_probs.device, dtype=torch.long)

        selected_probs = mask_probs[:, :phrase_len, :]
        token_probs = selected_probs.gather(
            dim=-1,
            index=token_ids.unsqueeze(0).unsqueeze(-1).expand(selected_probs.size(0), -1, 1)
        ).squeeze(-1)
        phrase_prob = token_probs.mean(dim=-1)
        return phrase_prob

    def _class_logits_from_verbalizer(self, mask_logits):
        mask_probs = F.softmax(mask_logits, dim=-1)
        class_scores = []

        for label in [0, 1]:
            phrase_scores = []
            for token_ids in self.resolved_verbalizer[label]:
                phrase_scores.append(self._phrase_probability(mask_probs, token_ids))
            phrase_scores = torch.stack(phrase_scores, dim=-1)
            class_score = phrase_scores.mean(dim=-1)
            class_scores.append(class_score)

        class_scores = torch.stack(class_scores, dim=-1)
        class_logits = torch.log(class_scores + 1e-12)
        return class_logits

    def forward(self, input_ids, attention_mask, mask_positions, labels=None):
        batch_size = input_ids.size(0)

        token_embeds = self.mlm.get_input_embeddings()(input_ids)
        soft_prompt = self.soft_prompt.unsqueeze(0).expand(batch_size, -1, -1)

        cls_embed = token_embeds[:, 0:1, :]
        rest_embeds = token_embeds[:, 1:, :]
        inputs_embeds = torch.cat([cls_embed, soft_prompt, rest_embeds], dim=1)

        cls_mask = attention_mask[:, 0:1]
        rest_mask = attention_mask[:, 1:]
        prompt_mask = self.soft_prompt_attention.unsqueeze(0).expand(batch_size, -1)
        ext_attention_mask = torch.cat([cls_mask, prompt_mask, rest_mask], dim=1)

        ext_mask_positions = mask_positions + self.num_soft_tokens

        backbone = self._get_backbone()
        outputs = backbone(
            inputs_embeds=inputs_embeds,
            attention_mask=ext_attention_mask,
            return_dict=True,
        )

        hidden = self.dropout(outputs.last_hidden_state)
        vocab_logits = self.mlm.cls(hidden)
        mask_logits = self._get_mask_logits(vocab_logits, ext_mask_positions)
        class_logits = self._class_logits_from_verbalizer(mask_logits)

        out = {
            "class_logits": class_logits,
            "mask_logits": mask_logits,
        }
        if labels is not None:
            out["ce_loss"] = F.cross_entropy(class_logits, labels)
        return out

In [419]:
def build_model_objects(dataset_type: str):
    tokenizer, resolved_verbalizer, num_label_masks, collator = build_tokenizer_and_collator(dataset_type)
    model = PaperSoftPromptDetector(
        model_name=infer_model_name(dataset_type),
        tokenizer=tokenizer,
        resolved_verbalizer=resolved_verbalizer,
        num_soft_tokens=GLOBAL_CONFIG["num_soft_tokens"],
        dropout=GLOBAL_CONFIG["dropout"],
        freeze_plm=GLOBAL_CONFIG["freeze_plm"],
    ).to(DEVICE)
    return tokenizer, resolved_verbalizer, num_label_masks, collator, model


In [420]:
def make_loader(df: pd.DataFrame, collator, batch_size: int, shuffle: bool, labeled: bool):
    dataset = TextFrameDataset(df, labeled=labeled)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=GLOBAL_CONFIG["num_workers"],
        pin_memory=GLOBAL_CONFIG["pin_memory"],
        collate_fn=collator,
    )

In [421]:
def l2_regularization(model: nn.Module):
    reg = torch.tensor(0.0, device=DEVICE)
    for p in model.parameters():
        if p.requires_grad:
            reg = reg + torch.sum(p ** 2)
    return reg


@torch.no_grad()
def create_ema_copy(model):
    """Create a frozen copy of the model for EMA teacher tracking."""
    ema = copy.deepcopy(model)
    for p in ema.parameters():
        p.requires_grad_(False)
    return ema


@torch.no_grad()
def update_ema(ema_model, model, decay=0.90):
    """Blend EMA parameters toward the current (fast) model."""
    for ema_p, model_p in zip(ema_model.parameters(), model.parameters()):
        ema_p.data.mul_(decay).add_(model_p.data, alpha=1.0 - decay)
    for ema_b, model_b in zip(ema_model.buffers(), model.buffers()):
        ema_b.data.copy_(model_b.data)


def adaptive_ema_decay(fast_acc: float, ema_prev_acc: float) -> float:
    """Adapt EMA decay based on accuracy delta between fast model and EMA teacher.

    If the fast model improves (delta > tau_up), lower decay toward ema_decay_min
    so the EMA absorbs gains faster.  If the fast model degrades (delta < -tau_down),
    raise decay toward ema_decay_max so the EMA resists noisy updates.
    Otherwise, use the base ema_decay.
    """
    base_decay = GLOBAL_CONFIG.get("ema_decay", 0.90)
    decay_min = GLOBAL_CONFIG.get("ema_decay_min", 0.70)
    decay_max = GLOBAL_CONFIG.get("ema_decay_max", 0.97)
    gamma = GLOBAL_CONFIG.get("ema_adapt_gamma", 1.5)
    tau_up = GLOBAL_CONFIG.get("ema_tau_up", 0.02)
    tau_down = GLOBAL_CONFIG.get("ema_tau_down", 0.01)

    delta = fast_acc - ema_prev_acc

    if delta > tau_up:
        # Fast model improved — lower decay to absorb gain faster
        decay = base_decay - gamma * (delta - tau_up)
    elif delta < -tau_down:
        # Fast model degraded — raise decay to resist noise
        decay = base_decay + gamma * (-delta - tau_down)
    else:
        decay = base_decay

    return float(max(decay_min, min(decay_max, decay)))

In [422]:
@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss, total_count = 0.0, 0

    autocast_enabled = torch.cuda.is_available() and GLOBAL_CONFIG["amp"]
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.amp.autocast(device_type="cuda", enabled=autocast_enabled):
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                mask_positions=batch["mask_positions"],
                labels=batch["labels"],
            )
        preds = torch.argmax(outputs["class_logits"], dim=-1)
        bs = batch["labels"].size(0)

        total_loss += outputs["ce_loss"].item() * bs
        total_count += bs
        all_preds.extend(preds.detach().cpu().tolist())
        all_labels.extend(batch["labels"].detach().cpu().tolist())

    return {
        "loss": total_loss / max(total_count, 1),
        "acc": accuracy_score(all_labels, all_preds),
        "f1": f1_score(all_labels, all_preds, zero_division=0),
    }

In [423]:
@torch.no_grad()
def predict_target_labels(model, df, collator, batch_size, desc="Predicting", temperature=None):
    """Predict pseudo-labels with configurable temperature scaling."""
    if temperature is None:
        temperature = GLOBAL_CONFIG.get("prediction_temperature", 1.0)

    model.eval()
    loader = make_loader(df, collator, batch_size=batch_size, shuffle=False, labeled=False)

    all_preds = []
    all_probs = []

    autocast_enabled = torch.cuda.is_available() and GLOBAL_CONFIG["amp"]
    pbar = tqdm(loader, desc=desc, leave=False)
    for batch in pbar:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.amp.autocast(device_type="cuda", enabled=autocast_enabled):
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                mask_positions=batch["mask_positions"],
            )

        scaled_logits = outputs["class_logits"] / temperature
        probs = torch.softmax(scaled_logits, dim=-1)
        max_probs, preds = torch.max(probs, dim=-1)

        all_preds.extend(preds.detach().cpu().tolist())
        all_probs.extend(max_probs.detach().cpu().tolist())
        pbar.set_postfix(mem=gpu_mem_text())

    return np.array(all_preds, dtype=int), np.array(all_probs, dtype=float)

In [424]:
def train_model(
    model, train_loader, epochs: int, lr: float, weight_decay: float, l2_alpha: float,
    task_name: str, stage_name: str, seed: int, iteration: int,
    anchor_state_dict=None, anchor_alpha: float = 0.0,
):
    """
    Train model with optional anchor regularization.

    When anchor_state_dict and anchor_alpha > 0, adds an L2 penalty
    that keeps model weights tethered to the source checkpoint,
    preventing catastrophic drift across sequential iterations.

    Pre-compute anchor reference pairs (param, ref) for efficiency.
    """
    anchor_pairs = []
    if anchor_state_dict is not None and anchor_alpha > 0:
        for name, param in model.named_parameters():
            if param.requires_grad and name in anchor_state_dict:
                anchor_pairs.append((param, anchor_state_dict[name].detach().to(DEVICE)))

    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr, weight_decay=weight_decay,
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs * len(train_loader), eta_min=lr * 0.1,
    )

    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available() and GLOBAL_CONFIG["amp"])
    accumulation_steps = max(1, GLOBAL_CONFIG["accumulation_steps"])
    grad_clip = GLOBAL_CONFIG["grad_clip"]

    history = []
    model.to(DEVICE)

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss, running_examples, running_correct = 0.0, 0, 0
        optimizer.zero_grad(set_to_none=True)

        desc = f"{stage_name} task={task_name} seed={seed} iter={iteration} epoch={epoch}/{epochs}"
        pbar = tqdm(train_loader, desc=desc, leave=False)

        for step_idx, batch in enumerate(pbar, start=1):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            autocast_enabled = torch.cuda.is_available() and GLOBAL_CONFIG["amp"]

            with torch.amp.autocast(device_type="cuda", enabled=autocast_enabled):
                outputs = model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    mask_positions=batch["mask_positions"],
                    labels=batch["labels"],
                )

                class_logits = outputs["class_logits"]
                labels = batch["labels"]

                # Eq 6: Cross-entropy loss
                ce_loss = F.cross_entropy(class_logits, labels)

                # Eq 7: L2 regularization
                reg = l2_regularization(model)

                # Anchor regularization – keeps model near source checkpoint
                anchor_loss = torch.tensor(0.0, device=DEVICE)
                if len(anchor_pairs) > 0:
                    for param, ref in anchor_pairs:
                        anchor_loss = anchor_loss + ((param - ref) ** 2).sum()

                loss = (ce_loss + l2_alpha * reg + anchor_alpha * anchor_loss) / accumulation_steps

            scaler.scale(loss).backward()

            did_step = 0
            if step_idx % accumulation_steps == 0 or step_idx == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                did_step = 1

            preds = torch.argmax(class_logits, dim=-1)
            bs = labels.size(0)

            running_loss    += loss.item() * accumulation_steps * bs
            running_examples += bs
            running_correct  += (preds == labels).sum().item()

            pbar.set_postfix(
                run_acc=f"{running_correct / max(running_examples, 1):.4f}",
                run_loss=f"{running_loss / max(running_examples, 1):.4f}",
                opt_step=did_step,
                mem=gpu_mem_text(),
            )

        epoch_row = {
            "task":       task_name,
            "seed":       seed,
            "stage":      stage_name,
            "iteration":  iteration,
            "epoch":      epoch,
            "train_loss": running_loss   / max(running_examples, 1),
            "train_acc":  running_correct / max(running_examples, 1),
        }
        history.append(epoch_row)

        print(f"{stage_name} iter={iteration} epoch={epoch}/{epochs} "
              f"train_loss={epoch_row['train_loss']:.6f} train_acc={epoch_row['train_acc']:.4f}")

        stage_thresholds = {
            "SOURCE_TRAIN": GLOBAL_CONFIG.get("early_stop_source_threshold", 1.0),
            "REFINE_SEQ":   GLOBAL_CONFIG.get("early_stop_refine_threshold",  0.95),
            "FINAL_TRAIN":  GLOBAL_CONFIG.get("early_stop_final_threshold",   0.97),
        }
        threshold = stage_thresholds.get(stage_name, 1.0)
        patience  = GLOBAL_CONFIG.get("early_stop_patience", 2)
        epoch_acc = running_correct / max(running_examples, 1)
        if epoch_acc >= threshold:
            recent = [row["train_acc"] for row in history[-patience:]]
            if len(recent) >= patience and all(v >= threshold for v in recent):
                print(
                    f"[EARLY STOPPED] {stage_name} iter={iteration} epoch={epoch}/{epochs} "
                    f"trainacc={epoch_acc:.4f} >= threshold={threshold:.2f} "
                    f"(patience={patience})"
                )
                break
      

    return model, pd.DataFrame(history)


In [425]:
def evaluate_full_target(model, target_df, collator):
    loader = make_loader(target_df[["model_text", "label"]].copy(), collator, GLOBAL_CONFIG["batch_size"], False, True)
    return evaluate_model(model, loader)


In [426]:
def split_indices_evenly(n: int, n_subsets: int, seed: int) -> List[np.ndarray]:
    rng = np.random.RandomState(seed)
    shuffled = rng.permutation(n)
    splits = np.array_split(shuffled, n_subsets)
    return [np.array(s, dtype=int) for s in splits]

In [427]:
def cns_weighted_vote(
    votes: List[List[int]],
    probs: List[List[float]],
    iter_indices: List[List[int]],
    beta: float,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """CNS-weighted voting for invariant label selection.

    For each sample, sort votes by iteration index, then compute a per-vote weight:
        weight = prob * (1 + beta * S)
    where S is the fraction of adjacent-iteration neighbours that agree with that vote.
    Winner = argmax of per-class weight sums; ties broken by mean confidence.

    Returns (labels, confidences, weighted_vote_ratios) as numpy arrays.
    """
    final_labels = []
    final_confidences = []
    vote_ratios = []

    for vote_list, prob_list, iter_list in zip(votes, probs, iter_indices):
        n = len(vote_list)
        if n == 0:
            final_labels.append(0)
            final_confidences.append(0.0)
            vote_ratios.append(0.0)
            continue

        # Sort by iteration index
        order = sorted(range(n), key=lambda k: iter_list[k])
        sorted_votes = [vote_list[k] for k in order]
        sorted_probs = [prob_list[k] for k in order]

        # Compute per-vote weights
        weights = [0.0] * n
        for j in range(n):
            neighbours = 0
            agree = 0
            if j > 0:
                neighbours += 1
                if sorted_votes[j - 1] == sorted_votes[j]:
                    agree += 1
            if j < n - 1:
                neighbours += 1
                if sorted_votes[j + 1] == sorted_votes[j]:
                    agree += 1
            s = agree / max(neighbours, 1)
            weights[j] = sorted_probs[j] * (1.0 + beta * s)

        # Aggregate per-class weight sums
        class_weights = {}
        class_probs = {}
        for j in range(n):
            c = sorted_votes[j]
            class_weights[c] = class_weights.get(c, 0.0) + weights[j]
            class_probs.setdefault(c, []).append(sorted_probs[j])

        # Winner = argmax of weight sums; break ties by mean confidence
        max_w = max(class_weights.values())
        candidates = [c for c, w in class_weights.items() if abs(w - max_w) < 1e-12]
        if len(candidates) == 1:
            winner = candidates[0]
        else:
            winner = max(candidates, key=lambda c: float(np.mean(class_probs[c])))

        winner_conf = float(np.mean(class_probs[winner]))
        total_weight = sum(class_weights.values())
        ratio = class_weights[winner] / total_weight if total_weight > 0 else 0.0

        final_labels.append(winner)
        final_confidences.append(winner_conf)
        vote_ratios.append(ratio)

    return (
        np.array(final_labels, dtype=int),
        np.array(final_confidences, dtype=float),
        np.array(vote_ratios, dtype=float),
    )


def majority_vote_with_confidence(
    votes: List[List[int]],
    probs: List[List[float]],
    iter_indices: List[List[int]] = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Backward-compatible wrapper around cns_weighted_vote.

    If iter_indices is not provided, falls back to sequential indices
    (0, 1, 2, ...) for each sample.
    """
    beta = GLOBAL_CONFIG.get("cns_beta", 0.5)
    if iter_indices is None:
        iter_indices = [list(range(len(v))) for v in votes]
    return cns_weighted_vote(votes, probs, iter_indices, beta)

In [428]:
def attach_vote_columns(df: pd.DataFrame, votes: List[List[int]], probs: List[List[float]],
                        iter_indices: List[List[int]] = None) -> pd.DataFrame:
    df = df.copy()
    df["vote_history_label"] = [json.dumps(v) for v in votes]
    df["vote_history_prob"] = [json.dumps([round(float(x), 6) for x in p]) for p in probs]
    df["vote_count"] = [len(v) for v in votes]
    if iter_indices is not None:
        df["vote_history_iter"] = [json.dumps(it) for it in iter_indices]
    return df

In [429]:
def build_refinement_train_df(
    initial_pseudo_df: pd.DataFrame,
    subset_indices: np.ndarray,
    source_df: pd.DataFrame,
    seed: int
):
    subset_df = initial_pseudo_df.iloc[subset_indices].copy().reset_index(drop=True)

    # 1. Filter out noisy predictions using confidence threshold
    conf_thresh = GLOBAL_CONFIG.get("train_refine_conf_threshold", 0.0)
    if conf_thresh > 0:
        confident_mask = subset_df["pseudo_conf"] >= conf_thresh
        subset_df = subset_df[confident_mask].copy()

    train_df = (
        subset_df[["model_text", "pseudo_label"]]
        .rename(columns={"pseudo_label": "label"})
        .copy()
    )

    # 2. SOURCE REPLAY: Mix high-quality source data to prevent catastrophic forgetting
    if GLOBAL_CONFIG.get("use_source_replay_in_refinement", True):
        factor = GLOBAL_CONFIG.get("source_replay_factor_refine", 4)
        source_repeated = pd.concat([source_df[["model_text", "label"]]] * factor, ignore_index=True)
        train_df = pd.concat([train_df, source_repeated], ignore_index=True)

    # Shuffle the combined dataset
    train_df = train_df.sample(frac=1.0, random_state=seed).reset_index(drop=True)

    info = {
        "subset_size": len(subset_indices),
        "confident_subset_size": len(subset_df),
        "refine_train_size": len(train_df),
    }
    return train_df, info

In [430]:
def select_invariant_samples(voted_target_df: pd.DataFrame):
    df = voted_target_df.copy().reset_index(drop=True)
    required_votes = GLOBAL_CONFIG["n_subsets"] - 1

    keep_rows = []
    pseudo_labels = []
    pseudo_confs = []
    vote_ratios = []

    # Fetch configuration or use your specified defaults
    req_ratio = GLOBAL_CONFIG.get("required_vote_ratio", 0.85) 
    final_conf_thresh = GLOBAL_CONFIG.get("final_invariant_conf_threshold", 0.70)
    beta = GLOBAL_CONFIG.get("cns_beta", 0.5)

    has_iter_col = "vote_history_iter" in df.columns

    for i in range(len(df)):
        vote_list = df.loc[i, "vote_history_label"]
        prob_list = df.loc[i, "vote_history_prob"]
        iter_list = None

        if isinstance(vote_list, str):
            vote_list = json.loads(vote_list)
        if isinstance(prob_list, str):
            prob_list = json.loads(prob_list)

        vote_list = [int(x) for x in vote_list]
        prob_list = [float(x) for x in prob_list]

        # Parse iteration indices if available
        if has_iter_col:
            raw_iter = df.loc[i, "vote_history_iter"]
            if isinstance(raw_iter, str):
                iter_list = [int(x) for x in json.loads(raw_iter)]
            elif isinstance(raw_iter, list):
                iter_list = [int(x) for x in raw_iter]

        # A sample must have been predicted in all held-out rounds
        if len(vote_list) != required_votes:
            print(f"  [WARN] Sample {i}: expected {required_votes} votes, got {len(vote_list)} — skipped")
            continue

        # Fall back to sequential indices if not available
        if iter_list is None:
            iter_list = list(range(len(vote_list)))

        # Use CNS-weighted vote for per-sample winner
        labels_arr, confs_arr, ratios_arr = cns_weighted_vote(
            [vote_list], [prob_list], [iter_list], beta
        )
        winner = int(labels_arr[0])
        conf = float(confs_arr[0])
        ratio = float(ratios_arr[0])

        # Keep if it meets the consensus and high confidence threshold
        if ratio >= req_ratio and conf >= final_conf_thresh:
            keep_rows.append(i)
            pseudo_labels.append(winner)
            pseudo_confs.append(conf)
            vote_ratios.append(ratio)

    invariant_df = df.iloc[keep_rows].copy().reset_index(drop=True)
    invariant_df["pseudo_label"] = np.array(pseudo_labels, dtype=int) if len(pseudo_labels) > 0 else []
    invariant_df["pseudo_conf"] = np.array(pseudo_confs, dtype=float) if len(pseudo_confs) > 0 else []
    invariant_df["vote_ratio"] = np.array(vote_ratios, dtype=float) if len(vote_ratios) > 0 else []

    stats = {
        "total_target_samples": len(df),
        "invariant_samples": len(invariant_df),
        "retention_rate": len(invariant_df) / max(len(df), 1),
        "invariant_pseudo_acc": (
            accuracy_score(invariant_df["gold_label"], invariant_df["pseudo_label"])
            if len(invariant_df) > 0 else np.nan
        ),
        "invariant_pseudo_f1": (
            f1_score(invariant_df["gold_label"], invariant_df["pseudo_label"], zero_division=0)
            if len(invariant_df) > 0 else np.nan
        ),
    }
    return invariant_df, stats

In [431]:
def run_source_training(task_name: str, seed: int):
    shots_per_class = GLOBAL_CONFIG["shots_per_class"]
    n_subsets = GLOBAL_CONFIG["n_subsets"]
    stage_banner(task_name, "SOURCE TRAINING", f"seed={seed} shots={shots_per_class}")

    seed_everything(seed)
    source_df, target_df, dataset_type = load_task_data(task_name, seed)
    tokenizer, resolved_verbalizer, num_label_masks, collator, model = build_model_objects(dataset_type)

    source_loader = make_loader(source_df[["model_text", "label"]], collator, GLOBAL_CONFIG["batch_size"], True, True)

    model, source_hist = train_model(
        model=model,
        train_loader=source_loader,
        epochs=GLOBAL_CONFIG["source_epochs"],
        lr=GLOBAL_CONFIG["lr"],
        weight_decay=GLOBAL_CONFIG["weight_decay"],
        l2_alpha=GLOBAL_CONFIG["l2_alpha"],
        task_name=task_name,
        stage_name="SOURCE_TRAIN",
        seed=seed,
        iteration=0,
    )

    # Save source model state dict for resetting during refinement
    source_state_dict = copy.deepcopy(model.state_dict())

    target_metrics = evaluate_full_target(model, target_df, collator)
    report_rows = [{
        "task": task_name,
        "seed": seed,
        "shots_per_class": shots_per_class,
        "n_subsets": n_subsets,
        "stage": "iter_0",
        "iteration": 0,
        "target_loss": target_metrics["loss"],
        "target_acc": target_metrics["acc"],
        "target_f1": target_metrics["f1"],
        "subset_size": np.nan,
        "confident_subset_size": np.nan,
        "refine_train_size": np.nan,
        "invariant_samples": np.nan,
        "retention_rate": np.nan,
    }]

    return {
        "source_df": source_df,
        "target_df": target_df,
        "dataset_type": dataset_type,
        "collator": collator,
        "model": model,
        "source_state_dict": source_state_dict,
        "source_history_df": source_hist,
        "report_rows": report_rows,
    }


In [432]:
def run_iterative_refinement(task_name: str, seed: int, source_df: pd.DataFrame,
                             target_df: pd.DataFrame, collator, model, source_state_dict):
    """
    Sequential iterative refinement following the paper's framework (θ₀ → θ₁ → ... → θ₈).
    Model weights are carried forward across iterations, benefiting from prior learning.

    Error propagation safeguards (without using ground truth):
    1. EMA Teacher: A slow-moving average of the model used for ALL predictions.
       Even if one bad iteration corrupts the fast model, the EMA barely moves,
       keeping pseudo-labels stable for subsequent iterations.
    2. Anchor Regularization: L2 penalty ||θ - θ₀||² toward source checkpoint
       prevents the model from drifting catastrophically far across iterations.
    3. Confidence-filtered pseudo-labels at the data level.
    4. Source replay mixed into every fold's training data.
    5. Adaptive EMA decay adjusts blending based on accuracy delta.
    6. CNS-weighted voting for invariant label selection.
    """
    stage_banner(task_name, "ITERATIVE REFINEMENT (SEQUENTIAL + EMA + ANCHOR + CNS)",
                 f"seed={seed} n_subsets={GLOBAL_CONFIG['n_subsets']}")

    df = target_df.copy().reset_index(drop=True)
    n_subsets = GLOBAL_CONFIG["n_subsets"]
    anchor_alpha = GLOBAL_CONFIG.get("anchor_alpha", 1e-3)
    beta = GLOBAL_CONFIG.get("cns_beta", 0.5)

    # --- Step 0: Get initial pseudo-labels from source model (θ₀) ---
    init_preds, init_probs = predict_target_labels(
        model, df, collator, GLOBAL_CONFIG["batch_size"],
        desc=f"INIT_PSEUDO | {task_name} | seed={seed}"
    )

    init_acc = accuracy_score(df["gold_label"].values, init_preds)
    print(f"--- [Source Model θ₀ Baseline] Pseudo-label acc on target: {init_acc:.4f} ---")

    df["init_pseudo_label"] = init_preds.astype(int)
    df["init_pseudo_conf"] = init_probs.astype(float)

    # Build DYNAMIC pseudo-label reference (updated by EMA teacher after each iteration)
    dynamic_pseudo_df = df[["model_text", "gold_label"]].copy()
    dynamic_pseudo_df["pseudo_label"] = init_preds.astype(int)
    dynamic_pseudo_df["pseudo_conf"] = init_probs.astype(float)

    # Create EMA teacher (starts as copy of source model θ₀)
    ema_model = create_ema_copy(model)

    # Split target into n_subsets folds
    subset_indices_list = split_indices_evenly(len(df), n_subsets, seed)
    all_indices = np.arange(len(df))

    # Accumulators for cross-fold votes
    votes = [[] for _ in range(len(df))]
    probs_votes = [[] for _ in range(len(df))]
    iter_votes = [[] for _ in range(len(df))]

    iter_rows = []
    refine_histories = []

    for iter_id, train_indices in enumerate(subset_indices_list, start=1):
        print(f"\n--- [{task_name}] Iteration {iter_id}/{n_subsets} (sequential θ_{iter_id-1} → θ_{iter_id}) ---")

        # 1. Build training data from DYNAMIC pseudo-labels for this fold + source replay
        refine_train_df, info = build_refinement_train_df(
            dynamic_pseudo_df, train_indices, source_df, seed
        )
        refine_loader = make_loader(refine_train_df, collator, GLOBAL_CONFIG["batch_size"], True, True)

        # 2. Train model SEQUENTIALLY (carries forward from previous iteration)
        #    Anchor regularization keeps weights tethered to source θ₀
        model, hist = train_model(
            model=model, train_loader=refine_loader,
            epochs=GLOBAL_CONFIG["refinement_epochs"],
            lr=GLOBAL_CONFIG["refinement_lr"],
            weight_decay=GLOBAL_CONFIG["weight_decay"],
            l2_alpha=GLOBAL_CONFIG["l2_alpha"],
            task_name=task_name, stage_name="REFINE_SEQ",
            seed=seed, iteration=iter_id,
            anchor_state_dict=source_state_dict,
            anchor_alpha=anchor_alpha,
        )

        if len(hist) > 0:
            hist = hist.copy()
            hist["subset_size"] = info["subset_size"]
            hist["confident_subset_size"] = info["confident_subset_size"]
            hist["refine_train_size"] = info["refine_train_size"]
            refine_histories.append(hist)

        # 3. Evaluate fast model and EMA model on full target
        target_metrics = evaluate_full_target(model, df, collator)
        ema_metrics = evaluate_full_target(ema_model, df, collator)

        # 4. Adaptive EMA decay: adjust based on fast vs EMA accuracy delta
        ema_prev_acc = ema_metrics["acc"]
        fast_acc = target_metrics["acc"]
        eff_decay = adaptive_ema_decay(fast_acc, ema_prev_acc)
        delta = fast_acc - ema_prev_acc
        print(f"  Adaptive EMA: fast_acc={fast_acc:.4f} ema_prev_acc={ema_prev_acc:.4f} "
              f"delta={delta:+.4f} eff_decay={eff_decay:.4f}")

        # 5. Update EMA teacher with adaptive decay
        update_ema(ema_model, model, eff_decay)

        # 6. Use EMA teacher (NOT fast model) to predict held-out folds → VOTES
        heldout_indices = np.setdiff1d(all_indices, train_indices)
        heldout_df = df.iloc[heldout_indices].copy().reset_index(drop=True)

        heldout_preds, heldout_probs = predict_target_labels(
            ema_model, heldout_df, collator, GLOBAL_CONFIG["batch_size"],
            desc=f"EMA_PREDICT_HELDOUT | {task_name} | iter={iter_id}"
        )

        # Collect votes from EMA predictions
        for local_i, global_i in enumerate(heldout_indices):
            votes[global_i].append(int(heldout_preds[local_i]))
            probs_votes[global_i].append(float(heldout_probs[local_i]))
            iter_votes[global_i].append(iter_id)

        # 7. Use EMA teacher to update dynamic pseudo-labels for ALL target data
        #    (next iteration trains on these updated, more stable labels)
        all_preds, all_probs_arr = predict_target_labels(
            ema_model, df, collator, GLOBAL_CONFIG["batch_size"],
            desc=f"EMA_UPDATE_PSEUDO | {task_name} | iter={iter_id}"
        )
        dynamic_pseudo_df["pseudo_label"] = all_preds.astype(int)
        dynamic_pseudo_df["pseudo_conf"] = all_probs_arr.astype(float)

        # Diagnostic: gold accuracy (for reporting only, never used for decisions)
        heldout_gold = df.iloc[heldout_indices]["gold_label"].values
        ema_fold_acc = accuracy_score(heldout_gold, heldout_preds)
        ema_class_ratio = float(np.mean(heldout_preds))
        overall_pseudo_acc = accuracy_score(df["gold_label"].values, all_preds)
        print(f"  EMA held-out acc: {ema_fold_acc:.4f} | class1 ratio: {ema_class_ratio:.2f} | overall pseudo acc: {overall_pseudo_acc:.4f}")

        iter_rows.append({
            "task": task_name, "seed": seed,
            "shots_per_class": GLOBAL_CONFIG["shots_per_class"],
            "n_subsets": n_subsets,
            "stage": f"iter_{iter_id}", "iteration": iter_id,
            "target_loss": target_metrics["loss"],
            "target_acc": target_metrics["acc"],
            "target_f1": target_metrics["f1"],
            "ema_target_acc": ema_metrics["acc"],
            "ema_target_f1": ema_metrics["f1"],
            "ema_decay_used": eff_decay,
            "subset_size": info["subset_size"],
            "confident_subset_size": info["confident_subset_size"],
            "refine_train_size": info["refine_train_size"],
            "invariant_samples": np.nan,
            "retention_rate": np.nan,
        })

    # --- Build final vote summary columns using CNS-weighted voting ---
    voted_labels, voted_confs, voted_ratios = cns_weighted_vote(
        votes, probs_votes, iter_votes, beta
    )

    # Rebuild per-sample arrays, falling back where no votes exist
    current_labels = []
    current_conf = []
    vote_ratio_list = []
    for i in range(len(df)):
        if len(votes[i]) == 0:
            current_labels.append(int(df.loc[i, "init_pseudo_label"]))
            current_conf.append(float(df.loc[i, "init_pseudo_conf"]))
            vote_ratio_list.append(np.nan)
        else:
            current_labels.append(int(voted_labels[i]))
            current_conf.append(float(voted_confs[i]))
            vote_ratio_list.append(float(voted_ratios[i]))

    df["current_pseudo_label"] = np.array(current_labels, dtype=int)
    df["current_pseudo_conf"] = np.array(current_conf, dtype=float)
    df["vote_ratio"] = np.array(vote_ratio_list, dtype=float)
    df["vote_count"] = [len(v) for v in votes]

    # Overall voted pseudo-label accuracy
    voted_acc = accuracy_score(df["gold_label"].values, df["current_pseudo_label"].values)
    print(f"\n--- [Voted Pseudo-label Accuracy] {voted_acc:.4f} ---")

    df = attach_vote_columns(df, votes, probs_votes, iter_votes)
    refine_history_df = pd.concat(refine_histories, axis=0, ignore_index=True) if refine_histories else pd.DataFrame()

    # Free EMA model memory
    del ema_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return model, df, pd.DataFrame(iter_rows), refine_history_df

In [433]:
def run_final_training(task_name: str, seed: int, source_df: pd.DataFrame,
                       voted_target_df: pd.DataFrame, collator, refined_model,
                       source_state_dict):
    """
    Final training on invariant pseudo-labels + source data.

    KEY FIX: Instead of training from a completely fresh model (which loses all
    task-specific knowledge), start from the SOURCE checkpoint. This preserves
    the detection features learned during source training while avoiding the
    corrupted weights from iterative refinement.
    """
    stage_banner(task_name, "FINAL TRAINING", f"seed={seed}")

    # 1. Apply voting rule to get invariant labels
    invariant_target_df, invariant_stats = select_invariant_samples(voted_target_df)

    print(f"  Invariant samples: {invariant_stats['invariant_samples']} / {invariant_stats['total_target_samples']}")
    print(f"  Retention rate: {invariant_stats['retention_rate']:.4f}")
    if not np.isnan(invariant_stats['invariant_pseudo_acc']):
        print(f"  Invariant pseudo-label accuracy: {invariant_stats['invariant_pseudo_acc']:.4f}")

    # 2. Combine invariant target labels + oversampled source labels
    final_train_pieces = []
    if len(invariant_target_df) > 0:
        inv_df = invariant_target_df[["model_text", "pseudo_label"]].rename(columns={"pseudo_label": "label"})
        final_train_pieces.append(inv_df)

    factor = GLOBAL_CONFIG.get("source_replay_factor_final", 8)
    source_repeated = pd.concat([source_df[["model_text", "label"]]] * factor, ignore_index=True)
    final_train_pieces.append(source_repeated)

    final_train_df = pd.concat(final_train_pieces, ignore_index=True).sample(
        frac=1.0, random_state=seed
    ).reset_index(drop=True)

    # 3. Start from SOURCE checkpoint (preserves task-specific features)
    dataset_type = TASK_REGISTRY[task_name]["dataset_type"]
    _, _, _, _, final_model = build_model_objects(dataset_type)
    final_model.load_state_dict(copy.deepcopy(source_state_dict))

    final_loader = make_loader(final_train_df, collator, GLOBAL_CONFIG["batch_size"], True, True)

    # 4. Train final model
    final_model, final_hist_df = train_model(
        model=final_model,
        train_loader=final_loader,
        epochs=GLOBAL_CONFIG["final_epochs"],
        lr=GLOBAL_CONFIG["lr"],
        weight_decay=GLOBAL_CONFIG["weight_decay"],
        l2_alpha=GLOBAL_CONFIG["l2_alpha"],
        task_name=task_name,
        stage_name="FINAL_TRAIN",
        seed=seed,
        iteration=9,
    )

    # 5. Evaluate on full target
    target_eval_df = voted_target_df[["model_text", "gold_label"]].rename(columns={"gold_label": "label"})
    target_metrics = evaluate_full_target(final_model, target_eval_df, collator)

    final_row = {
        "task": task_name,
        "seed": seed,
        "shots_per_class": GLOBAL_CONFIG["shots_per_class"],
        "n_subsets": GLOBAL_CONFIG["n_subsets"],
        "stage": "final",
        "iteration": 9,
        "target_loss": target_metrics["loss"],
        "target_acc": target_metrics["acc"],
        "target_f1": target_metrics["f1"],
        "subset_size": np.nan,
        "confident_subset_size": np.nan,
        "refine_train_size": len(final_train_df),
        "invariant_samples": invariant_stats["invariant_samples"],
        "retention_rate": invariant_stats["retention_rate"],
    }

    print(f"[FINAL RESULT] target_acc={target_metrics['acc']:.4f} target_f1={target_metrics['f1']:.4f}")

    return final_model, final_hist_df, final_row, final_train_df, invariant_target_df, invariant_stats

In [434]:
def final_pipeline(task_name: str, seed: int):
    if task_name not in TASK_REGISTRY:
        raise ValueError(f"Unknown task_name={task_name}. Available: {list(TASK_REGISTRY.keys())}")

    GLOBAL_CONFIG["seed"] = int(seed)

    n_subsets = int(GLOBAL_CONFIG["n_subsets"])
    shots_per_class = int(GLOBAL_CONFIG["shots_per_class"])

    if n_subsets <= 0:
        raise ValueError("n_subsets must be >= 1")
    if shots_per_class <= 0:
        raise ValueError("shots_per_class must be >= 1")

    run_prefix = task_prefix(task_name, seed)

    stage_banner(
        task_name,
        "FINAL PIPELINE START",
        f"seed={seed} shots={shots_per_class} n_subsets={n_subsets}"
    )

    # save exact run config
    run_config = copy.deepcopy(GLOBAL_CONFIG)
    run_config["task_name"] = task_name
    run_config["source_file"] = TASK_REGISTRY[task_name]["source_file"]
    run_config["target_file"] = TASK_REGISTRY[task_name]["target_file"]
    run_config["dataset_type"] = TASK_REGISTRY[task_name]["dataset_type"]

    config_path = os.path.join(GLOBAL_CONFIG["save_configs_dir"], f"{run_prefix}_config.json")
    save_json(run_config, config_path)

    # ---------------------------
    # 1) Source training
    # ---------------------------
    source_out = run_source_training(
        task_name=task_name,
        seed=seed,
    )

    source_df = source_out["source_df"]
    target_df = source_out["target_df"]
    collator = source_out["collator"]
    model = source_out["model"]
    source_state_dict = source_out["source_state_dict"]
    source_history_df = source_out["source_history_df"]
    report_rows = list(source_out["report_rows"])

    # save source checkpoint immediately
    source_ckpt_path = os.path.join(
        GLOBAL_CONFIG["save_checkpoints_dir"],
        f"{run_prefix}_iter0_source_model.pt"
    )
    torch.save(source_state_dict, source_ckpt_path)

    # ---------------------------
    # 2) Iterative refinement (independent models per fold)
    # ---------------------------
    refined_model, voted_target_df, iter_metrics_df, refine_history_df = run_iterative_refinement(
        task_name=task_name,
        seed=seed,
        source_df=source_df,
        target_df=target_df,
        collator=collator,
        model=model,
        source_state_dict=source_state_dict,
    )

    if len(iter_metrics_df) > 0:
        report_rows.extend(iter_metrics_df.to_dict("records"))

    refined_ckpt_path = os.path.join(
        GLOBAL_CONFIG["save_checkpoints_dir"],
        f"{run_prefix}_refined_model.pt"
    )
    torch.save(refined_model.state_dict(), refined_ckpt_path)

    # ---------------------------
    # 3) Final training (from source checkpoint)
    # ---------------------------
    final_model, final_hist_df, final_row, final_train_df, invariant_target_df, invariant_stats = run_final_training(
        task_name=task_name,
        seed=seed,
        source_df=source_df,
        voted_target_df=voted_target_df,
        collator=collator,
        refined_model=refined_model,
        source_state_dict=source_state_dict,
    )
    report_rows.append(final_row)

    final_ckpt_path = os.path.join(
        GLOBAL_CONFIG["save_checkpoints_dir"],
        f"{run_prefix}_final_model.pt"
    )
    torch.save(final_model.state_dict(), final_ckpt_path)

    # ---------------------------
    # 4) Save reports
    # ---------------------------
    report_df = pd.DataFrame(report_rows)
    report_df = report_df.sort_values("iteration").reset_index(drop=True)

    main_report_path = os.path.join(
        GLOBAL_CONFIG["save_reports_dir"],
        f"{run_prefix}.csv"
    )
    save_csv(report_df, main_report_path)

    source_hist_path = os.path.join(
        GLOBAL_CONFIG["save_histories_dir"],
        f"{run_prefix}_source_history.csv"
    )
    save_csv(source_history_df, source_hist_path)

    refine_hist_path = os.path.join(
        GLOBAL_CONFIG["save_histories_dir"],
        f"{run_prefix}_refine_history.csv"
    )
    if len(refine_history_df) > 0:
        save_csv(refine_history_df, refine_hist_path)
    else:
        pd.DataFrame().to_csv(refine_hist_path, index=False)

    final_hist_path = os.path.join(
        GLOBAL_CONFIG["save_histories_dir"],
        f"{run_prefix}_final_history.csv"
    )
    save_csv(final_hist_df, final_hist_path)

    voted_target_path = os.path.join(
        GLOBAL_CONFIG["save_votes_dir"],
        f"{run_prefix}_voted_target.csv"
    )
    save_csv(voted_target_df, voted_target_path)

    invariant_target_path = os.path.join(
        GLOBAL_CONFIG["save_votes_dir"],
        f"{run_prefix}_invariant_target.csv"
    )
    save_csv(invariant_target_df, invariant_target_path)

    # one-row summary
    summary_df = pd.DataFrame([{
        "task": task_name,
        "seed": seed,
        "shots_per_class": shots_per_class,
        "n_subsets": n_subsets,
        "iter0_target_acc": report_df.loc[report_df["stage"] == "iter_0", "target_acc"].iloc[0] if (report_df["stage"] == "iter_0").any() else np.nan,
        "iter0_target_f1": report_df.loc[report_df["stage"] == "iter_0", "target_f1"].iloc[0] if (report_df["stage"] == "iter_0").any() else np.nan,
        "best_mid_iter_acc": report_df[report_df["stage"].str.startswith("iter_")]["target_acc"].max() if len(report_df) > 0 else np.nan,
        "best_mid_iter_f1": report_df[report_df["stage"].str.startswith("iter_")]["target_f1"].max() if len(report_df) > 0 else np.nan,
        "final_target_acc": final_row["target_acc"],
        "final_target_f1": final_row["target_f1"],
        "invariant_samples": invariant_stats["invariant_samples"],
        "retention_rate": invariant_stats["retention_rate"],
        "invariant_pseudo_acc": invariant_stats["invariant_pseudo_acc"],
        "invariant_pseudo_f1": invariant_stats["invariant_pseudo_f1"],
        "main_report_csv": main_report_path,
        "source_history_csv": source_hist_path,
        "refine_history_csv": refine_hist_path,
        "final_history_csv": final_hist_path,
        "voted_target_csv": voted_target_path,
        "invariant_target_csv": invariant_target_path,
        "source_ckpt": source_ckpt_path,
        "refined_ckpt": refined_ckpt_path,
        "final_ckpt": final_ckpt_path,
        "config_json": config_path,
    }])

    summary_path = os.path.join(
        GLOBAL_CONFIG["save_reports_dir"],
        f"{run_prefix}_summary.csv"
    )
    save_csv(summary_df, summary_path)

    # ---------------------------
    # 5) Console display
    # ---------------------------
    stage_banner(task_name, "RUN COMPLETE", f"seed={seed}")

    print("\nMain per-iteration report:")
    display(report_df)

    print("\nRun summary:")
    display(summary_df)

    print("\nSaved files:")
    print(" main_report_path   :", main_report_path)
    print(" summary_path       :", summary_path)
    print(" source_hist_path   :", source_hist_path)
    print(" refine_hist_path   :", refine_hist_path)
    print(" final_hist_path    :", final_hist_path)
    print(" voted_target_path  :", voted_target_path)
    print(" invariant_target   :", invariant_target_path)
    print(" source_ckpt_path   :", source_ckpt_path)
    print(" refined_ckpt_path  :", refined_ckpt_path)
    print(" final_ckpt_path    :", final_ckpt_path)
    print(" config_path        :", config_path)

    out = {
        "task": task_name,
        "seed": seed,
        "shots_per_class": shots_per_class,
        "n_subsets": n_subsets,
        "report_df": report_df,
        "summary_df": summary_df,
        "source_history_df": source_history_df,
        "refine_history_df": refine_history_df,
        "final_history_df": final_hist_df,
        "voted_target_df": voted_target_df,
        "invariant_target_df": invariant_target_df,
        "final_train_df": final_train_df,
        "invariant_stats": invariant_stats,
        "paths": {
            "main_report_csv": main_report_path,
            "summary_csv": summary_path,
            "source_history_csv": source_hist_path,
            "refine_history_csv": refine_hist_path,
            "final_history_csv": final_hist_path,
            "voted_target_csv": voted_target_path,
            "invariant_target_csv": invariant_target_path,
            "source_ckpt": source_ckpt_path,
            "refined_ckpt": refined_ckpt_path,
            "final_ckpt": final_ckpt_path,
            "config_json": config_path,
        },
        "final_model": final_model,
    }

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return out

In [435]:
def print_run_settings():
    print("=" * 100)
    print("Current GLOBAL_CONFIG runtime settings")
    print("=" * 100)
    print("seed                :", GLOBAL_CONFIG["seed"])
    print("shots_per_class     :", GLOBAL_CONFIG["shots_per_class"])
    print("n_subsets           :", GLOBAL_CONFIG["n_subsets"])
    print("batch_size          :", GLOBAL_CONFIG["batch_size"])
    print("source_epochs       :", GLOBAL_CONFIG["source_epochs"])
    print("refinement_epochs   :", GLOBAL_CONFIG["refinement_epochs"])
    print("final_epochs        :", GLOBAL_CONFIG["final_epochs"])
    print("lr                  :", GLOBAL_CONFIG["lr"])
    print("refinement_lr       :", GLOBAL_CONFIG["refinement_lr"])
    print("ema_decay           :", GLOBAL_CONFIG["ema_decay"])
    print("anchor_alpha        :", GLOBAL_CONFIG["anchor_alpha"])
    print("prediction_temp     :", GLOBAL_CONFIG["prediction_temperature"])
    print("dropout             :", GLOBAL_CONFIG["dropout"])
    print("freeze_plm          :", GLOBAL_CONFIG["freeze_plm"])
    print("train_conf_thresh   :", GLOBAL_CONFIG["train_refine_conf_threshold"])
    print("final_conf_thresh   :", GLOBAL_CONFIG["final_invariant_conf_threshold"])
    print("required_vote_ratio :", GLOBAL_CONFIG["required_vote_ratio"])
    print("replay_refine       :", GLOBAL_CONFIG["source_replay_factor_refine"])
    print("replay_final        :", GLOBAL_CONFIG["source_replay_factor_final"])
    print("label_map           :", GLOBAL_CONFIG["label_map"])
    print("=" * 100)

In [436]:
def inspect_and_run(
    task_name: str,
    seed: int,
    inspect_only: bool = False,
):
    print_run_settings()
    inspect_task_data(task_name, seed=seed)

    if inspect_only:
        print("\ninspect_only=True, so training was not started.")
        return None

    return final_pipeline(
        task_name=task_name,
        seed=seed,
    )



In [437]:
import os

for key in ["save_reports_dir", "save_histories_dir", "save_checkpoints_dir", "save_votes_dir", "save_configs_dir"]:
    os.makedirs(GLOBAL_CONFIG[key], exist_ok=True)
    print(f"{GLOBAL_CONFIG[key]}")

/nfsshare/users/P127003093/Mini Project/output/reports
/nfsshare/users/P127003093/Mini Project/output/histories
/nfsshare/users/P127003093/Mini Project/output/checkpoints
/nfsshare/users/P127003093/Mini Project/output/votes
/nfsshare/users/P127003093/Mini Project/output/configs


In [438]:
result = final_pipeline(task_name="P->O", seed=41)
clear_gpu_memory()


[TASK=P->O] [FINAL PIPELINE START] DEVICE=cuda:0 GPU_MEM=1.21G/1.21G seed=41 shots=25 n_subsets=8

[TASK=P->O] [SOURCE TRAINING] DEVICE=cuda:0 GPU_MEM=1.21G/1.21G seed=41 shots=25


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-chinese
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


SOURCE_TRAIN task=P->O seed=41 iter=0 epoch=1/20:   0%|          | 0/2 [00:00<?, ?it/s]

SOURCE_TRAIN iter=0 epoch=1/20 train_loss=193.584725 train_acc=0.4800


...

FINAL_TRAIN iter=9 epoch=4/20 train_loss=165.175623 train_acc=0.9963
[EARLY STOPPED] FINAL_TRAIN iter=9 epoch=4/20 trainacc=0.9963 >= threshold=0.97 (patience=2)
[FINAL RESULT] target_acc=0.8720 target_f1=0.8813

[TASK=P->O] [RUN COMPLETE] DEVICE=cuda:0 GPU_MEM=2.35G/7.63G seed=41

Main per-iteration report:


,task,seed,shots_per_class,n_subsets,stage,iteration,target_loss,target_acc,target_f1,subset_size,confident_subset_size,refine_train_size,invariant_samples,retention_rate,ema_target_acc,ema_target_f1,ema_decay_used
0,P->O,41,25,8,iter_0,0,1.373265,0.8695,0.856357,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,P->O,41,25,8,iter_1,1,0.118684,0.9790,0.978809,250.0,243.0,543.0,NaN,NaN,0.8695,0.856357,0.9500
2,P->O,41,25,8,iter_2,2,0.362034,0.9265,0.923398,250.0,244.0,544.0,NaN,NaN,0.9005,0.900251,0.9585
3,P->O,41,25,8,iter_3,3,0.629743,0.8845,0.883978,250.0,244.0,544.0,NaN,NaN,0.8820,0.887189,0.9750
4,P->O,41,25,8,iter_4,4,0.715074,0.9140,0.913741,250.0,246.0,546.0,NaN,NaN,0.8660,0.874883,0.9500
5,P->O,41,25,8,iter_5,5,0.613472,0.8805,0.886784,250.0,242.0,542.0,NaN,NaN,0.8485,0.862585,0.9500
6,P->O,41,25,8,iter_6,6,1.097352,0.8575,0.869446,250.0,234.0,534.0,NaN,NaN,0.8295,0.848780,0.9555
7,P->O,41,25,8,iter_7,7,0.926186,0.8375,0.853933,250.0,234.0,534.0,NaN,NaN,0.8250,0.845815,0.9750
8,P->O,41,25,8,iter_8,8,0.945273,0.8600,0.871205,250.0,232.0,532.0,NaN,NaN,0.8235,0.844835,0.9500
9,P->O,41,25,8,final,9,1.132518,0.8720,0.881262,NaN,NaN,1916.0,1666.0,0.833,NaN,NaN,NaN



Run summary:


,task,seed,shots_per_class,n_subsets,iter0_target_acc,iter0_target_f1,best_mid_iter_acc,best_mid_iter_f1,final_target_acc,final_target_f1,...,main_report_csv,source_history_csv,refine_history_csv,final_history_csv,voted_target_csv,invariant_target_csv,source_ckpt,refined_ckpt,final_ckpt,config_json
0,P->O,41,25,8,0.8695,0.856357,0.979,0.978809,0.872,0.881262,...,/nfsshare/users/P127003093/Mini Project/output...,/nfsshare/users/P127003093/Mini Project/output...,/nfsshare/users/P127003093/Mini Project/output...,/nfsshare/users/P127003093/Mini Project/output...,/nfsshare/users/P127003093/Mini Project/output...,/nfsshare/users/P127003093/Mini Project/output...,/nfsshare/users/P127003093/Mini Project/output...,/nfsshare/users/P127003093/Mini Project/output...,/nfsshare/users/P127003093/Mini Project/output...,/nfsshare/users/P127003093/Mini Project/output...



Saved files:
 main_report_path   : /nfsshare/users/P127003093/Mini Project/output/reports/P_O_41.csv
 summary_path       : /nfsshare/users/P127003093/Mini Project/output/reports/P_O_41_summary.csv
 source_hist_path   : /nfsshare/users/P127003093/Mini Project/output/histories/P_O_41_source_history.csv
 refine_hist_path   : /nfsshare/users/P127003093/Mini Project/output/histories/P_O_41_refine_history.csv
 final_hist_path    : /nfsshare/users/P127003093/Mini Project/output/histories/P_O_41_final_history.csv
 voted_target_path  : /nfsshare/users/P127003093/Mini Project/output/votes/P_O_41_voted_target.csv
 invariant_target   : /nfsshare/users/P127003093/Mini Project/output/votes/P_O_41_invariant_target.csv
 source_ckpt_path   : /nfsshare/users/P127003093/Mini Project/output/checkpoints/P_O_41_iter0_source_model.pt
 refined_ckpt_path  : /nfsshare/users/P127003093/Mini Project/output/checkpoints/P_O_41_refined_model.pt
 final_ckpt_path    : /nfsshare/users/P127003093/Mini Project/output/ch

In [439]:
result = final_pipeline(task_name="P->O", seed=10)
clear_gpu_memory()


[TASK=P->O] [FINAL PIPELINE START] DEVICE=cuda:0 GPU_MEM=1.21G/1.26G seed=10 shots=25 n_subsets=8

[TASK=P->O] [SOURCE TRAINING] DEVICE=cuda:0 GPU_MEM=1.21G/1.26G seed=10 shots=25


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-chinese
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


SOURCE_TRAIN task=P->O seed=10 iter=0 epoch=1/20:   0%|          | 0/2 [00:00<?, ?it/s]

SOURCE_TRAIN iter=0 epoch=1/20 train_loss=193.654858 train_acc=0.5400


...

FINAL_TRAIN iter=9 epoch=4/20 train_loss=168.046092 train_acc=0.9736
[EARLY STOPPED] FINAL_TRAIN iter=9 epoch=4/20 trainacc=0.9736 >= threshold=0.97 (patience=2)
[FINAL RESULT] target_acc=0.9170 target_f1=0.9223

[TASK=P->O] [RUN COMPLETE] DEVICE=cuda:0 GPU_MEM=2.35G/7.63G seed=10

Main per-iteration report:


,task,seed,shots_per_class,n_subsets,stage,iteration,target_loss,target_acc,target_f1,subset_size,confident_subset_size,refine_train_size,invariant_samples,retention_rate,ema_target_acc,ema_target_f1,ema_decay_used
0,P->O,10,25,8,iter_0,0,0.540649,0.8935,0.901889,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,P->O,10,25,8,iter_1,1,0.634709,0.9125,0.910026,250.0,244.0,544.0,NaN,NaN,0.8935,0.901889,0.96900
2,P->O,10,25,8,iter_2,2,0.384797,0.9380,0.941288,250.0,244.0,544.0,NaN,NaN,0.9015,0.908329,0.95000
3,P->O,10,25,8,iter_3,3,0.564596,0.9305,0.934834,250.0,238.0,538.0,NaN,NaN,0.9050,0.911463,0.95925
4,P->O,10,25,8,iter_4,4,0.987044,0.8620,0.877224,250.0,247.0,547.0,NaN,NaN,0.9025,0.909429,0.99990
5,P->O,10,25,8,iter_5,5,0.579924,0.9200,0.924670,250.0,238.0,538.0,NaN,NaN,0.9025,0.909429,0.97125
6,P->O,10,25,8,iter_6,6,0.365141,0.9230,0.928439,250.0,244.0,544.0,NaN,NaN,0.9050,0.911463,0.97050
7,P->O,10,25,8,iter_7,7,0.679597,0.9020,0.910584,250.0,241.0,541.0,NaN,NaN,0.9000,0.907236,0.97500
8,P->O,10,25,8,iter_8,8,0.389365,0.9045,0.911778,250.0,245.0,545.0,NaN,NaN,0.8960,0.903882,0.97500
9,P->O,10,25,8,final,9,0.496094,0.9170,0.922285,NaN,NaN,2157.0,1907.0,0.9535,NaN,NaN,NaN



Run summary:


,task,seed,shots_per_class,n_subsets,iter0_target_acc,iter0_target_f1,best_mid_iter_acc,best_mid_iter_f1,final_target_acc,final_target_f1,...,main_report_csv,source_history_csv,refine_history_csv,final_history_csv,voted_target_csv,invariant_target_csv,source_ckpt,refined_ckpt,final_ckpt,config_json
0,P->O,10,25,8,0.8935,0.901889,0.938,0.941288,0.917,0.922285,...,/nfsshare/users/P127003093/Mini Project/output...,/nfsshare/users/P127003093/Mini Project/output...,/nfsshare/users/P127003093/Mini Project/output...,/nfsshare/users/P127003093/Mini Project/output...,/nfsshare/users/P127003093/Mini Project/output...,/nfsshare/users/P127003093/Mini Project/output...,/nfsshare/users/P127003093/Mini Project/output...,/nfsshare/users/P127003093/Mini Project/output...,/nfsshare/users/P127003093/Mini Project/output...,/nfsshare/users/P127003093/Mini Project/output...



Saved files:
 main_report_path   : /nfsshare/users/P127003093/Mini Project/output/reports/P_O_10.csv
 summary_path       : /nfsshare/users/P127003093/Mini Project/output/reports/P_O_10_summary.csv
 source_hist_path   : /nfsshare/users/P127003093/Mini Project/output/histories/P_O_10_source_history.csv
 refine_hist_path   : /nfsshare/users/P127003093/Mini Project/output/histories/P_O_10_refine_history.csv
 final_hist_path    : /nfsshare/users/P127003093/Mini Project/output/histories/P_O_10_final_history.csv
 voted_target_path  : /nfsshare/users/P127003093/Mini Project/output/votes/P_O_10_voted_target.csv
 invariant_target   : /nfsshare/users/P127003093/Mini Project/output/votes/P_O_10_invariant_target.csv
 source_ckpt_path   : /nfsshare/users/P127003093/Mini Project/output/checkpoints/P_O_10_iter0_source_model.pt
 refined_ckpt_path  : /nfsshare/users/P127003093/Mini Project/output/checkpoints/P_O_10_refined_model.pt
 final_ckpt_path    : /nfsshare/users/P127003093/Mini Project/output/ch

In [440]:
result = final_pipeline(task_name="P->O", seed=140)
clear_gpu_memory()


[TASK=P->O] [FINAL PIPELINE START] DEVICE=cuda:0 GPU_MEM=1.21G/1.26G seed=140 shots=25 n_subsets=8

[TASK=P->O] [SOURCE TRAINING] DEVICE=cuda:0 GPU_MEM=1.21G/1.26G seed=140 shots=25


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-chinese
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


SOURCE_TRAIN task=P->O seed=140 iter=0 epoch=1/20:   0%|          | 0/2 [00:00<?, ?it/s]

SOURCE_TRAIN iter=0 epoch=1/20 train_loss=193.555675 train_acc=0.5400


...

FINAL_TRAIN iter=9 epoch=3/20 train_loss=169.553885 train_acc=0.9903
[EARLY STOPPED] FINAL_TRAIN iter=9 epoch=3/20 trainacc=0.9903 >= threshold=0.97 (patience=2)
[FINAL RESULT] target_acc=0.9655 target_f1=0.9667

[TASK=P->O] [RUN COMPLETE] DEVICE=cuda:0 GPU_MEM=2.35G/7.63G seed=140

Main per-iteration report:


,task,seed,shots_per_class,n_subsets,stage,iteration,target_loss,target_acc,target_f1,subset_size,confident_subset_size,refine_train_size,invariant_samples,retention_rate,ema_target_acc,ema_target_f1,ema_decay_used
0,P->O,140,25,8,iter_0,0,0.116312,0.9615,0.962712,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,P->O,140,25,8,iter_1,1,0.115921,0.9725,0.973131,250.0,239.0,539.0,NaN,NaN,0.9615,0.962712,0.97500
2,P->O,140,25,8,iter_2,2,0.138045,0.9805,0.980873,250.0,235.0,535.0,NaN,NaN,0.9635,0.964580,0.97200
3,P->O,140,25,8,iter_3,3,0.262971,0.9660,0.967118,250.0,244.0,544.0,NaN,NaN,0.9655,0.966489,0.97500
4,P->O,140,25,8,iter_4,4,0.593849,0.8850,0.896861,250.0,243.0,543.0,NaN,NaN,0.9665,0.967460,0.99990
5,P->O,140,25,8,iter_5,5,0.389688,0.9560,0.957814,250.0,241.0,541.0,NaN,NaN,0.9665,0.967460,0.97575
6,P->O,140,25,8,iter_6,6,0.156357,0.9700,0.970845,250.0,242.0,542.0,NaN,NaN,0.9680,0.968962,0.97500
7,P->O,140,25,8,iter_7,7,0.071050,0.9880,0.988131,250.0,242.0,542.0,NaN,NaN,0.9695,0.970374,0.96975
8,P->O,140,25,8,iter_8,8,0.205809,0.9755,0.976086,250.0,246.0,546.0,NaN,NaN,0.9730,0.973659,0.97500
9,P->O,140,25,8,final,9,0.297428,0.9655,0.966651,NaN,NaN,2165.0,1915.0,0.9575,NaN,NaN,NaN



Run summary:


,task,seed,shots_per_class,n_subsets,iter0_target_acc,iter0_target_f1,best_mid_iter_acc,best_mid_iter_f1,final_target_acc,final_target_f1,...,main_report_csv,source_history_csv,refine_history_csv,final_history_csv,voted_target_csv,invariant_target_csv,source_ckpt,refined_ckpt,final_ckpt,config_json
0,P->O,140,25,8,0.9615,0.962712,0.988,0.988131,0.9655,0.966651,...,/nfsshare/users/P127003093/Mini Project/output...,/nfsshare/users/P127003093/Mini Project/output...,/nfsshare/users/P127003093/Mini Project/output...,/nfsshare/users/P127003093/Mini Project/output...,/nfsshare/users/P127003093/Mini Project/output...,/nfsshare/users/P127003093/Mini Project/output...,/nfsshare/users/P127003093/Mini Project/output...,/nfsshare/users/P127003093/Mini Project/output...,/nfsshare/users/P127003093/Mini Project/output...,/nfsshare/users/P127003093/Mini Project/output...



Saved files:
 main_report_path   : /nfsshare/users/P127003093/Mini Project/output/reports/P_O_140.csv
 summary_path       : /nfsshare/users/P127003093/Mini Project/output/reports/P_O_140_summary.csv
 source_hist_path   : /nfsshare/users/P127003093/Mini Project/output/histories/P_O_140_source_history.csv
 refine_hist_path   : /nfsshare/users/P127003093/Mini Project/output/histories/P_O_140_refine_history.csv
 final_hist_path    : /nfsshare/users/P127003093/Mini Project/output/histories/P_O_140_final_history.csv
 voted_target_path  : /nfsshare/users/P127003093/Mini Project/output/votes/P_O_140_voted_target.csv
 invariant_target   : /nfsshare/users/P127003093/Mini Project/output/votes/P_O_140_invariant_target.csv
 source_ckpt_path   : /nfsshare/users/P127003093/Mini Project/output/checkpoints/P_O_140_iter0_source_model.pt
 refined_ckpt_path  : /nfsshare/users/P127003093/Mini Project/output/checkpoints/P_O_140_refined_model.pt
 final_ckpt_path    : /nfsshare/users/P127003093/Mini Project/

In [441]:
#result = final_pipeline(task_name="M->S", seed=41)
#clear_gpu_memory()

In [442]:
#result = final_pipeline(task_name="M->S", seed=10)
#clear_gpu_memory()

In [443]:
#result = final_pipeline(task_name="M->S", seed=140)

In [444]:
"""import datetime
import traceback

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ---------------------------------------------------------------------------
# NFS-safe storage (no SQLite)
# ---------------------------------------------------------------------------
STUDY_DIR = os.path.expanduser("~/Mini Project")
os.makedirs(STUDY_DIR, exist_ok=True)
JOURNAL_PATH = os.path.join(STUDY_DIR, "optuna_study.log")
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# ---------------------------------------------------------------------------
# Module-level trial log for crash-safe JSON backup
# ---------------------------------------------------------------------------
TRIAL_LOG: list = []
TRIAL_LOG_PATH = os.path.join(STUDY_DIR, "optuna_trial_log.json")

# Reload from disk if resuming after a kernel restart
if os.path.exists(TRIAL_LOG_PATH):
    with open(TRIAL_LOG_PATH, "r") as _f:
        TRIAL_LOG = json.load(_f)
    print(f"Resumed {len(TRIAL_LOG)} previous trial records from {TRIAL_LOG_PATH}")

# ---------------------------------------------------------------------------
# Search configuration
# ---------------------------------------------------------------------------
SEARCH_TASK = "A->M"
SEARCH_SEEDS = [41, 42, 1234]

SEARCHED_KEYS = [
    "lr", "refinement_lr", "dropout", "num_soft_tokens", "l2_alpha",
    "weight_decay", "ema_decay", "anchor_alpha", "cns_beta",
    "train_refine_conf_threshold", "required_vote_ratio",
    "source_replay_factor_refine", "ema_adapt_gamma",
]


def suggest_params(trial: optuna.Trial) -> dict:
    return {
        "lr":                        trial.suggest_float("lr", 1e-5, 5e-5, log=True),
        "refinement_lr":             trial.suggest_float("refinement_lr", 5e-6, 3e-5, log=True),
        "dropout":                   trial.suggest_float("dropout", 0.3, 0.6),
        "num_soft_tokens":           trial.suggest_int("num_soft_tokens", 5, 20),
        "l2_alpha":                  trial.suggest_float("l2_alpha", 1e-4, 1e-2, log=True),
        "weight_decay":              trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True),
        "ema_decay":                 trial.suggest_float("ema_decay", 0.85, 0.95),
        "anchor_alpha":              trial.suggest_float("anchor_alpha", 1e-4, 1e-2, log=True),
        "cns_beta":                  trial.suggest_float("cns_beta", 0.1, 1.5),
        "train_refine_conf_threshold": trial.suggest_float("train_refine_conf_threshold", 0.50, 0.75),
        "required_vote_ratio":       trial.suggest_float("required_vote_ratio", 0.75, 1.0),
        "source_replay_factor_refine": trial.suggest_int("source_replay_factor_refine", 1, 6),
        "ema_adapt_gamma":           trial.suggest_float("ema_adapt_gamma", 0.5, 3.0),
    }


def _flush_trial_log():
    """Atomically write TRIAL_LOG to JSON (write-then-rename for NFS safety)."""
    tmp = TRIAL_LOG_PATH + ".tmp"
    with open(tmp, "w") as f:
        json.dump(TRIAL_LOG, f, indent=2)
    os.replace(tmp, TRIAL_LOG_PATH)


# Snapshot original config so every trial starts from a clean slate
_ORIGINAL_CONFIG = copy.deepcopy(GLOBAL_CONFIG)


def objective(trial: optuna.Trial) -> float:
    # --- 1. Suggest and patch ------------------------------------------------
    params = suggest_params(trial)
    config_backup = copy.deepcopy(GLOBAL_CONFIG)

    try:
        for k, v in params.items():
            GLOBAL_CONFIG[k] = v

        seed_f1s = []
        seed_accs = []

        for seed_idx, seed in enumerate(SEARCH_SEEDS):
            seed_everything(seed)
            clear_gpu_memory()

            result = final_pipeline(task_name=SEARCH_TASK, seed=seed)

            final_f1 = float(result["summary_df"]["final_target_f1"].iloc[0])
            final_acc = float(result["summary_df"]["final_target_acc"].iloc[0])
            seed_f1s.append(final_f1)
            seed_accs.append(final_acc)

            # Free model memory between seeds
            del result
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            running_mean_f1 = float(np.mean(seed_f1s))
            trial.report(running_mean_f1, step=seed_idx)

            print(f"  Trial {trial.number} | seed={seed} | f1={final_f1:.4f} | "
                  f"acc={final_acc:.4f} | running_mean_f1={running_mean_f1:.4f}")

            # Partial-result backup after each seed
            partial_entry = {
                "trial": trial.number,
                "status": "partial" if seed_idx < len(SEARCH_SEEDS) - 1 else "complete",
                "seed": seed,
                "seed_idx": seed_idx,
                "f1": final_f1,
                "acc": final_acc,
                "running_mean_f1": running_mean_f1,
                "params": {k: (int(v) if isinstance(v, np.generic) else v)
                           for k, v in params.items()},
                "timestamp": datetime.datetime.now().isoformat(),
            }
            TRIAL_LOG.append(partial_entry)
            _flush_trial_log()

            if trial.should_prune():
                raise optuna.TrialPruned(
                    f"Pruned after seed_idx={seed_idx} (mean_f1={running_mean_f1:.4f})"
                )

        # --- 2. Final metrics -------------------------------------------------
        mean_f1 = float(np.mean(seed_f1s))
        std_f1 = float(np.std(seed_f1s))
        mean_acc = float(np.mean(seed_accs))

        trial.set_user_attr("seed_f1s", seed_f1s)
        trial.set_user_attr("seed_accs", seed_accs)
        trial.set_user_attr("mean_f1", mean_f1)
        trial.set_user_attr("std_f1", std_f1)
        trial.set_user_attr("mean_acc", mean_acc)

        # Final completed-trial log entry
        TRIAL_LOG.append({
            "trial": trial.number,
            "status": "complete_summary",
            "seed_f1s": seed_f1s,
            "seed_accs": seed_accs,
            "mean_f1": mean_f1,
            "std_f1": std_f1,
            "mean_acc": mean_acc,
            "params": {k: (int(v) if isinstance(v, np.generic) else v)
                       for k, v in params.items()},
            "timestamp": datetime.datetime.now().isoformat(),
        })
        _flush_trial_log()

        return mean_f1

    except optuna.TrialPruned:
        raise  # let Optuna handle pruned trials
    except Exception as e:
        TRIAL_LOG.append({
            "trial": trial.number,
            "status": "error",
            "error": str(e),
            "traceback": traceback.format_exc(),
            "params": {k: (int(v) if isinstance(v, np.generic) else v)
                       for k, v in params.items()},
            "timestamp": datetime.datetime.now().isoformat(),
        })
        _flush_trial_log()
        raise
    finally:
        # Always restore GLOBAL_CONFIG to its pre-trial state
        GLOBAL_CONFIG.clear()
        GLOBAL_CONFIG.update(config_backup)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=42, n_startup_trials=10),
    pruner=MedianPruner(n_startup_trials=8, n_warmup_steps=1, interval_steps=1),
    study_name="mini_project_3seed_search",
    storage=storage,
    load_if_exists=True,
)

print(f"Study '{study.study_name}' created/resumed.  "
      f"Existing trials: {len(study.trials)}")

# ---------------------------------------------------------------------------
# Run optimisation
# ---------------------------------------------------------------------------
study.optimize(
    objective,
    n_trials=60,
    timeout=12 * 3600,
    gc_after_trial=True,
    show_progress_bar=True,
)

# ===========================================================================
# Post-search analysis
# ===========================================================================
completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
pruned = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]

print("\n" + "=" * 100)
print("OPTUNA SEARCH COMPLETE")
print("=" * 100)
print(f"  Completed trials : {len(completed)}")
print(f"  Pruned trials    : {len(pruned)}")

# --- Best trial ---------------------------------------------------------------
if not completed:
    print("No trials completed successfully. "
          "Check TRIAL_LOG for errors before proceeding.")
    print(f"Error log: {TRIAL_LOG_PATH}")
else:
    best = study.best_trial
print(f"\n{'─'*80}")
print(f"BEST TRIAL: #{best.number}")
print(f"  mean_f1  = {best.user_attrs['mean_f1']:.4f}")
print(f"  std_f1   = {best.user_attrs['std_f1']:.4f}")
print(f"  mean_acc = {best.user_attrs['mean_acc']:.4f}")
print(f"  seed_f1s = {best.user_attrs['seed_f1s']}")
print(f"  seed_accs= {best.user_attrs['seed_accs']}")

print(f"\n  {'Parameter':<35s} {'Old':>12s} {'New':>12s}")
print(f"  {'─'*35} {'─'*12} {'─'*12}")
for k in SEARCHED_KEYS:
    old_val = _ORIGINAL_CONFIG.get(k, "N/A")
    new_val = best.params.get(k, "N/A")
    if isinstance(old_val, float):
        print(f"  {k:<35s} {old_val:>12.6g} {new_val:>12.6g}")
    else:
        print(f"  {k:<35s} {str(old_val):>12s} {str(new_val):>12s}")

# --- Top-5 trials -------------------------------------------------------------
sorted_completed = sorted(completed, key=lambda t: t.value, reverse=True)
top5 = sorted_completed[:5]

print(f"\n{'─'*80}")
print("TOP-5 TRIALS (by mean_f1):")
print(f"  {'#':>5s}  {'mean_f1':>8s}  {'std_f1':>8s}  {'mean_acc':>8s}")
for t in top5:
    mf = t.user_attrs.get("mean_f1", t.value)
    sf = t.user_attrs.get("std_f1", float("nan"))
    ma = t.user_attrs.get("mean_acc", float("nan"))
    print(f"  {t.number:5d}  {mf:8.4f}  {sf:8.4f}  {ma:8.4f}")

# --- Stability recommendation -------------------------------------------------
if top5:
    most_stable = min(top5, key=lambda t: t.user_attrs.get("std_f1", float("inf")))
    if most_stable.number != best.number:
        ms_mf = most_stable.user_attrs.get("mean_f1", most_stable.value)
        ms_sf = most_stable.user_attrs.get("std_f1", float("nan"))
        print(f"\n  NOTE: Trial #{most_stable.number} (mean_f1={ms_mf:.4f}, std_f1={ms_sf:.4f}) "
              f"is the most stable among the top-5. Consider it if robustness matters.")
else:
    print("\n  NOTE: Fewer than 5 trials completed — stability comparison skipped.")

# ---------------------------------------------------------------------------
# Lock best params into GLOBAL_CONFIG and save
# ---------------------------------------------------------------------------
for k, v in best.params.items():
    GLOBAL_CONFIG[k] = v

best_hparams_record = {
    "study_name": study.study_name,
    "best_trial": best.number,
    "best_value": best.value,
    "mean_f1": best.user_attrs.get("mean_f1"),
    "std_f1": best.user_attrs.get("std_f1"),
    "mean_acc": best.user_attrs.get("mean_acc"),
    "seed_f1s": best.user_attrs.get("seed_f1s"),
    "seed_accs": best.user_attrs.get("seed_accs"),
    "seeds": SEARCH_SEEDS,
    "task": SEARCH_TASK,
    "params": dict(best.params),
    "n_completed": len(completed),
    "n_pruned": len(pruned),
    "timestamp": datetime.datetime.now().isoformat(),
}

path = os.path.join(
    GLOBAL_CONFIG["save_configs_dir"],
    f"best_hparams_{datetime.datetime.now():%Y%m%d_%H%M%S}.json",
)
save_json(best_hparams_record, path)
print(f"\nBest hyperparameters saved to: {path}")

print("\nGLOBAL_CONFIG locked. Ready for full sweep.")"""

SyntaxError: invalid syntax (3093024915.py, line 64)